In [1]:
import pandas as pd
import numpy as np
from helpers import Load_model
from scipy import stats
import matplotlib.pyplot as plt

In [2]:
model, tokenizer, frames, threshold = Load_model("best")

Model, tokenizer, frames, and threshold loaded from: best
Using device: cuda


In [4]:
df = pd.read_parquet("data/merged_topic_and_frames_filtered.parquet")

# Uncomment to include new data sources 
#df2 = pd.read_parquet("data/merged_topic_and_frames_filtered_p2.parquet")
#df = pd.concat([df1, df2], axis=0, ignore_index=True)

In [5]:
# add Datetime cols for analysis 
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df["year_month"] = df["date"].dt.to_period("M").astype(str)
df["quarter"] = df["date"].dt.to_period("Q").astype(str)

# Make each frame its own column
frame_matrix = np.array(df['vector'].tolist())
for i, frame in enumerate(frames):
    df[f'frame_{frame}'] = frame_matrix[:, i]
frame_cols = [f'frame_{f}' for f in frames]

df.to_parquet("data/merged_topic_and_frames_filtered_with_time_and_frames.parquet", index=False)
    
df.head(1).T

,0
uuid,11362746988981500303
url,https://www.huffpost.com/entry/cancer-debunkin...
outlet_name,HuffPost
bias,Left
date,2017-03-13 00:00:00
content,The term “integrative therapies” describes the...
content_preprocessed,The term “integrative therapies” describes the...
topic_top1,healthcare
topic_top1_score,0.981742
topic_top2,Russia


In [6]:
outlets = df["outlet_name"].unique()

annual_dict = {}
quarterly_dict = {}
monthly_dict = {}

In [7]:
outlets

array(['HuffPost', 'Newsweek', 'The Washington Post', 'BBC News', 'CNBC',
       'Reuters', 'New York Post', 'Breitbart News', 'Washington Times'],
      dtype=object)

In [8]:
# Annual frame distribution
for outlet in outlets:
    temp = df[df["outlet_name"] == outlet].copy()
    yearly = temp.groupby("year")[frame_cols].mean() * 100 
    yearly.index.name = f"{outlet}"
    annual_dict[outlet] = yearly
    display(yearly.round(2).T)


HuffPost,2015,2016,2017,2018,2019,2020,2021
frame_cap&res,10.21,11.21,12.07,6.06,5.51,3.99,2.42
frame_crime,15.46,17.78,19.72,29.30,29.66,22.92,38.15
frame_culture,37.31,40.99,39.16,32.54,29.56,23.88,24.76
frame_economic,20.66,20.32,23.02,13.38,13.13,10.25,8.32
frame_fairness,40.12,41.19,41.28,49.01,47.19,43.70,38.78
frame_health,24.54,25.86,22.96,20.85,17.94,34.62,24.76
frame_legality,35.34,31.68,36.82,53.66,56.01,55.20,62.49
frame_morality,5.37,5.19,5.98,5.63,3.81,3.03,3.48
frame_policy,58.69,52.79,58.10,52.82,53.21,52.79,48.58
frame_political,11.76,15.33,15.25,19.30,29.06,26.63,24.03


Newsweek,2015,2016,2017,2018,2019,2020,2021
frame_cap&res,4.71,4.62,3.86,3.37,3.49,3.28,4.59
frame_crime,39.96,36.04,44.04,39.88,38.97,31.48,38.30
frame_culture,23.89,22.35,19.36,21.76,22.29,17.77,21.75
frame_economic,16.00,14.69,11.69,9.50,11.01,9.81,11.18
frame_fairness,36.57,38.67,35.22,33.16,36.73,42.56,38.50
frame_health,15.86,11.18,14.80,19.18,19.03,31.96,24.03
frame_legality,70.64,66.37,75.84,66.46,67.11,67.56,66.68
frame_morality,3.12,2.47,3.54,4.30,3.99,3.14,2.73
frame_policy,58.80,58.66,58.13,57.95,59.25,63.25,59.31
frame_political,22.51,29.64,29.40,24.31,28.10,27.92,21.28


The Washington Post,2015,2016,2017,2018,2019,2020,2021
frame_cap&res,5.79,6.00,6.10,19.35,10.30,19.09,30.54
frame_crime,24.59,23.67,28.57,25.81,20.00,18.67,12.10
frame_culture,21.49,26.42,21.78,20.43,21.82,14.52,23.65
frame_economic,18.60,16.98,19.34,27.42,18.79,24.90,39.66
frame_fairness,45.25,54.03,51.05,29.57,33.33,36.10,29.61
frame_health,15.50,13.38,12.72,17.74,20.61,25.31,17.50
frame_legality,58.06,60.21,67.25,44.09,36.97,38.59,34.82
frame_morality,3.51,3.95,2.61,4.84,3.03,2.49,2.23
frame_policy,59.71,60.21,61.67,46.24,48.48,46.89,56.42
frame_political,30.58,42.88,37.46,24.19,20.00,19.92,19.74


BBC News,2015,2016,2017,2018,2019,2020,2021
frame_cap&res,4.36,3.54,15.87,3.27,3.62,4.27,2.99
frame_crime,15.38,18.07,15.87,20.26,32.57,20.73,25.64
frame_culture,13.85,10.61,20.63,32.03,34.87,40.24,29.49
frame_economic,12.31,16.31,23.81,16.99,17.43,21.34,20.09
frame_fairness,21.41,18.07,19.05,16.99,12.83,12.20,11.11
frame_health,14.87,16.31,14.29,20.26,15.46,25.61,16.24
frame_legality,37.69,41.85,38.10,43.14,20.72,20.12,23.08
frame_morality,0.38,0.39,1.59,0.00,1.64,1.83,1.28
frame_policy,68.59,71.12,66.67,63.40,54.28,61.59,53.42
frame_political,3.33,4.52,7.94,6.54,2.96,3.05,4.70


CNBC,2015,2016,2017,2018,2019,2020,2021
frame_cap&res,41.26,36.16,34.37,35.06,36.46,36.11,38.37
frame_crime,5.94,7.64,8.93,10.82,9.37,5.97,7.72
frame_culture,9.44,11.16,11.04,9.58,10.08,7.46,7.82
frame_economic,83.22,73.55,71.71,72.60,72.52,67.39,70.51
frame_fairness,14.86,20.87,17.99,15.67,17.95,18.67,18.30
frame_health,8.22,7.44,9.31,6.76,7.72,24.43,18.51
frame_legality,43.71,39.88,43.05,48.93,48.82,46.45,45.94
frame_morality,0.00,1.03,0.50,0.56,0.39,0.26,0.10
frame_policy,93.71,88.02,91.56,93.35,93.70,93.88,94.48
frame_political,6.99,13.02,12.03,10.26,10.55,7.46,7.17


Reuters,2015,2016,2017,2018,2019,2020,2021
frame_cap&res,32.79,30.24,30.05,29.28,33.58,31.06,37.36
frame_crime,17.49,22.71,19.35,18.05,16.83,11.89,12.64
frame_culture,6.56,4.28,3.97,6.26,5.78,3.85,3.76
frame_economic,53.55,50.15,48.32,51.93,53.69,47.27,57.06
frame_fairness,20.22,20.65,21.03,21.92,20.60,16.07,14.01
frame_health,13.11,12.83,12.14,13.54,12.31,36.42,24.83
frame_legality,67.76,71.98,67.31,66.85,68.01,61.28,58.09
frame_morality,1.09,0.74,0.36,0.74,0.59,0.38,0.23
frame_policy,87.43,83.78,84.13,85.82,88.69,85.20,88.95
frame_political,10.38,12.54,15.50,12.15,11.81,6.77,5.01


New York Post,2015,2016,2017,2018,2019,2020,2021
frame_cap&res,4.49,4.02,8.30,5.00,4.35,4.90,5.02
frame_crime,40.90,37.05,42.08,47.08,43.86,33.12,38.64
frame_culture,28.93,26.56,21.04,16.98,22.54,18.62,20.59
frame_economic,20.45,21.43,21.24,18.39,16.97,15.29,16.26
frame_fairness,32.92,34.60,37.26,35.66,32.38,31.21,29.29
frame_health,11.97,14.06,16.41,17.83,18.97,31.65,25.29
frame_legality,54.11,57.14,62.93,65.61,60.57,58.79,60.21
frame_morality,2.49,1.79,1.93,1.97,2.52,2.30,1.84
frame_policy,41.90,42.63,50.77,41.51,43.95,52.52,50.65
frame_political,12.97,18.97,15.25,16.07,15.06,16.81,16.01


Breitbart News,2015,2016,2017,2018,2019,2020,2021
frame_cap&res,3.46,2.05,2.12,2.34,3.20,3.14,4.18
frame_crime,47.80,45.18,47.08,49.45,45.99,41.78,45.83
frame_culture,17.49,16.80,12.36,14.00,13.03,11.13,9.37
frame_economic,10.57,7.59,10.69,8.51,10.43,9.21,9.99
frame_fairness,48.64,52.83,50.87,52.34,53.17,50.46,49.31
frame_health,7.20,5.97,6.60,5.55,7.18,18.56,15.88
frame_legality,72.50,73.06,80.89,81.50,82.44,78.53,80.24
frame_morality,6.08,7.34,5.53,7.15,7.92,5.22,5.03
frame_policy,51.64,49.66,54.59,50.55,54.65,53.73,55.77
frame_political,36.30,44.37,36.47,39.58,44.37,36.46,33.06


Washington Times,2015,2016,2017,2018,2019,2020,2021
frame_cap&res,3.93,2.27,3.46,2.74,4.17,5.92,4.36
frame_crime,42.79,39.67,41.84,45.94,45.22,33.77,34.07
frame_culture,13.68,15.50,12.00,12.39,13.48,11.09,9.36
frame_economic,12.81,11.16,12.11,10.86,13.48,12.49,13.07
frame_fairness,36.68,45.87,33.73,44.52,48.04,42.24,47.85
frame_health,18.20,12.19,18.38,12.61,13.60,25.31,21.39
frame_legality,80.79,80.37,75.89,82.57,83.82,78.55,80.95
frame_morality,1.16,4.13,1.95,1.43,2.94,1.40,2.28
frame_policy,63.17,61.98,60.00,60.20,60.05,62.94,66.58
frame_political,19.80,36.98,19.57,33.44,36.03,29.75,31.14


In [9]:
# Quarterly frame distribution
for outlet in outlets:
    temp = df[df["outlet_name"] == outlet].copy()
    quarterly = temp.groupby("quarter")[frame_cols].mean() * 100 
    quarterly.index.name = f"{outlet}"
    quarterly_dict[outlet] = quarterly
    display(quarterly.round(2).T)

HuffPost,2015Q1,2015Q2,2015Q3,2015Q4,2016Q1,2016Q2,2016Q3,2016Q4,2017Q1,2017Q2,...,2019Q3,2019Q4,2020Q1,2020Q2,2020Q3,2020Q4,2021Q1,2021Q2,2021Q3,2021Q4
frame_cap&res,7.25,10.10,10.65,12.18,9.04,12.48,13.03,10.15,10.35,10.97,...,3.40,5.51,5.08,4.46,5.05,2.85,1.73,5.26,2.08,2.54
frame_crime,15.36,13.94,14.41,18.16,16.11,16.76,16.23,22.41,16.36,21.10,...,30.19,33.07,16.95,27.51,21.14,24.09,49.71,63.16,31.25,35.12
frame_culture,38.55,38.46,34.45,38.39,41.85,36.72,45.49,40.38,38.10,39.66,...,31.32,33.07,25.42,21.56,28.08,22.19,21.39,26.32,18.75,25.95
frame_economic,17.97,22.60,19.83,21.84,19.25,21.39,23.05,17.34,21.53,20.89,...,13.21,11.81,13.98,7.81,9.46,10.30,7.51,10.53,10.42,8.32
frame_fairness,37.68,42.79,41.34,38.16,37.72,40.82,41.08,45.45,41.41,37.97,...,49.81,44.09,38.56,44.24,44.16,45.17,39.88,42.11,39.58,38.36
frame_health,26.96,19.71,26.51,25.06,27.50,25.31,27.05,23.47,23.60,24.89,...,16.60,16.14,38.98,44.61,29.34,31.38,23.70,0.00,29.17,25.39
frame_legality,36.81,37.02,36.53,31.26,31.43,31.02,28.06,36.58,34.58,33.97,...,57.74,53.15,51.69,54.65,49.53,59.59,71.10,63.16,68.75,59.94
frame_morality,4.93,4.57,5.85,5.98,3.93,5.17,4.81,6.98,5.59,6.12,...,4.91,2.76,1.69,4.46,3.15,2.85,3.47,10.53,2.08,3.39
frame_policy,54.78,62.50,59.29,57.47,56.58,52.05,52.51,49.89,55.49,57.17,...,51.70,54.33,61.44,46.10,51.74,52.93,45.09,36.84,60.42,48.94
frame_political,7.83,13.22,14.82,10.11,14.54,13.19,16.03,17.97,16.15,14.56,...,35.47,29.53,18.64,19.70,25.87,32.96,38.15,21.05,16.67,21.16


Newsweek,2015Q1,2015Q2,2015Q3,2015Q4,2016Q1,2016Q2,2016Q3,2016Q4,2017Q1,2017Q2,...,2019Q3,2019Q4,2020Q1,2020Q2,2020Q3,2020Q4,2021Q1,2021Q2,2021Q3,2021Q4
frame_cap&res,5.60,6.10,3.21,4.15,3.72,5.83,6.15,2.21,4.47,4.03,...,4.46,2.97,2.41,5.13,3.27,2.30,3.81,3.91,5.64,4.77
frame_crime,40.41,39.33,34.69,44.24,36.99,32.76,38.30,37.29,39.43,43.26,...,37.89,44.27,33.08,31.25,31.37,30.79,40.04,41.29,33.64,38.10
frame_culture,25.37,29.88,23.32,18.66,22.30,21.61,23.40,22.38,18.90,17.67,...,25.41,16.97,19.55,17.97,19.61,14.98,16.03,23.00,25.69,21.95
frame_economic,15.93,19.21,13.12,15.90,13.94,16.30,15.13,12.71,13.01,13.80,...,12.78,10.75,8.12,12.05,8.06,10.48,12.22,11.58,9.15,11.34
frame_fairness,40.41,32.62,41.69,32.49,34.20,38.94,39.72,43.65,38.01,35.50,...,37.89,36.07,38.95,38.84,42.16,48.16,47.12,40.89,29.85,37.42
frame_health,14.16,18.29,17.78,13.82,12.45,11.15,9.69,11.05,9.15,15.19,...,22.44,18.67,30.68,44.20,31.81,22.79,18.95,20.93,29.67,25.15
frame_legality,74.63,69.82,69.97,68.66,66.54,64.67,65.72,69.61,79.47,75.19,...,64.78,68.03,61.80,64.73,67.43,73.53,76.53,68.61,58.23,65.27
frame_morality,4.72,2.44,4.08,1.61,2.79,1.54,2.84,3.04,1.83,2.64,...,4.46,2.12,3.16,3.68,4.14,1.84,3.37,4.07,1.57,2.37
frame_policy,55.46,59.45,60.93,59.22,59.29,59.52,55.56,59.94,62.40,61.40,...,54.38,59.83,65.56,67.08,63.18,58.73,59.34,53.04,59.33,61.81
frame_political,24.19,14.94,25.07,24.88,26.39,27.79,31.21,35.64,31.50,29.30,...,26.45,28.01,28.27,14.29,22.98,43.11,36.05,23.40,11.65,18.42


The Washington Post,2015Q1,2015Q2,2015Q3,2015Q4,2016Q1,2016Q2,2016Q3,2016Q4,2017Q1,2017Q2,...,2019Q3,2019Q4,2020Q1,2020Q2,2020Q3,2020Q4,2021Q1,2021Q2,2021Q3,2021Q4
frame_cap&res,6.61,5.22,5.93,5.41,6.85,6.21,7.30,3.87,5.23,5.30,...,4.35,2.38,16.67,19.61,22.62,15.79,16.49,25.64,26.87,35.93
frame_crime,22.31,26.12,24.58,25.23,21.23,19.31,29.20,25.16,30.07,29.55,...,21.74,28.57,20.00,23.53,21.43,11.84,17.53,17.95,11.94,9.88
frame_culture,17.36,17.91,22.88,28.83,24.66,29.66,25.55,25.81,19.61,16.67,...,19.57,23.81,16.67,9.80,16.67,14.47,22.68,30.77,22.39,23.35
frame_economic,19.83,20.15,18.64,15.32,19.86,24.14,13.14,10.97,15.69,15.91,...,8.70,9.52,16.67,21.57,32.14,22.37,20.62,35.90,35.82,46.41
frame_fairness,44.63,45.52,41.53,49.55,56.16,57.93,52.55,49.68,50.98,50.00,...,36.96,35.71,10.00,35.29,45.24,36.84,41.24,28.21,32.84,25.75
frame_health,10.74,17.16,16.95,17.12,13.70,9.66,14.60,15.48,15.03,11.36,...,21.74,19.05,33.33,27.45,26.19,19.74,17.53,20.51,23.88,15.87
frame_legality,58.68,57.46,54.24,62.16,56.16,56.55,64.96,63.23,70.59,65.91,...,50.00,40.48,36.67,27.45,40.48,44.74,39.18,33.33,31.34,34.43
frame_morality,3.31,2.99,5.93,1.80,1.37,4.14,5.11,5.16,1.31,1.52,...,4.35,2.38,0.00,3.92,3.57,1.32,3.09,2.56,0.00,2.40
frame_policy,60.33,61.94,55.93,60.36,53.42,61.38,61.31,64.52,67.97,60.61,...,47.83,38.10,46.67,49.02,45.24,47.37,42.27,53.85,55.22,61.08
frame_political,28.93,32.09,30.51,30.63,39.73,40.69,41.61,49.03,37.25,43.18,...,17.39,28.57,3.33,11.76,22.62,28.95,26.80,28.21,20.90,16.47


BBC News,2015Q1,2015Q2,2015Q3,2015Q4,2016Q1,2016Q2,2016Q3,2016Q4,2017Q1,2017Q2,...,2019Q3,2019Q4,2020Q1,2020Q2,2020Q3,2020Q4,2021Q1,2021Q2,2021Q3,2021Q4
frame_cap&res,2.33,4.69,4.61,4.60,3.31,3.61,2.0,9.52,15.0,25.00,...,3.48,0.00,3.23,3.03,0.00,6.94,3.19,5.88,0.00,1.72
frame_crime,8.14,14.06,16.59,16.46,18.75,18.67,10.0,23.81,25.0,16.67,...,36.52,19.05,25.81,33.33,10.71,16.67,18.09,39.22,41.94,17.24
frame_culture,12.79,26.56,11.52,13.32,8.09,8.43,24.0,28.57,15.0,16.67,...,37.83,33.33,54.84,27.27,57.14,33.33,26.60,21.57,29.03,41.38
frame_economic,15.12,6.25,13.82,11.86,18.75,12.65,16.0,14.29,25.0,25.00,...,17.83,23.81,22.58,6.06,32.14,23.61,26.60,13.73,32.26,8.62
frame_fairness,8.14,17.19,20.28,25.42,18.75,12.65,34.0,14.29,25.0,25.00,...,10.87,9.52,9.68,15.15,7.14,13.89,7.45,15.69,3.23,17.24
frame_health,18.60,21.88,16.59,12.11,14.71,17.47,18.0,23.81,20.0,33.33,...,13.04,23.81,22.58,48.48,14.29,20.83,17.02,31.37,9.68,5.17
frame_legality,40.70,32.81,39.17,37.05,40.44,46.39,38.0,33.33,35.0,41.67,...,14.78,23.81,16.13,18.18,14.29,25.00,24.47,29.41,16.13,18.97
frame_morality,0.00,1.56,0.92,0.00,0.00,0.60,2.0,0.00,5.0,0.00,...,2.17,0.00,3.23,3.03,3.57,0.00,0.00,0.00,6.45,1.72
frame_policy,76.74,57.81,68.20,68.77,72.43,73.49,68.0,42.86,75.0,58.33,...,51.30,57.14,74.19,72.73,53.57,54.17,69.15,64.71,32.26,29.31
frame_political,2.33,4.69,2.76,3.63,3.68,3.61,8.0,14.29,10.0,16.67,...,3.48,0.00,3.23,0.00,3.57,4.17,4.26,7.84,3.23,3.45


CNBC,2015Q1,2015Q2,2015Q3,2015Q4,2016Q1,2016Q2,2016Q3,2016Q4,2017Q1,2017Q2,...,2019Q3,2019Q4,2020Q1,2020Q2,2020Q3,2020Q4,2021Q1,2021Q2,2021Q3,2021Q4
frame_cap&res,39.76,42.61,43.33,38.13,46.51,34.02,34.65,29.94,32.98,36.45,...,37.46,36.58,39.20,38.82,36.00,34.16,35.35,36.76,38.70,42.81
frame_crime,6.02,4.78,5.00,8.63,6.20,4.12,7.92,10.83,10.11,7.88,...,12.07,9.44,6.82,6.25,7.38,5.09,10.57,7.35,8.47,4.21
frame_culture,13.25,8.26,6.67,11.51,6.98,13.40,11.88,12.74,14.89,12.81,...,10.53,12.98,7.10,7.89,7.08,7.58,8.46,8.82,6.50,7.19
frame_economic,86.75,86.09,83.33,76.26,77.52,78.35,75.25,66.24,70.21,71.43,...,70.28,72.86,70.17,66.45,66.77,66.87,66.77,70.59,70.62,74.74
frame_fairness,14.46,12.17,16.67,17.99,13.95,12.37,22.77,30.57,18.09,15.27,...,17.03,16.22,12.50,20.72,18.77,20.25,18.73,20.83,19.21,15.44
frame_health,9.64,7.83,9.17,7.19,6.20,8.25,6.93,8.28,9.04,9.85,...,6.50,7.37,24.15,29.28,19.69,24.61,17.22,18.87,19.21,19.30
frame_legality,38.55,47.83,35.00,47.48,37.21,34.02,43.56,43.31,46.81,39.90,...,49.54,44.84,46.02,46.38,50.77,45.17,48.64,40.20,48.59,45.26
frame_morality,0.00,0.00,0.00,0.00,1.55,1.03,0.00,1.27,0.00,0.49,...,0.62,0.29,0.28,0.33,0.00,0.31,0.30,0.00,0.00,0.00
frame_policy,95.18,96.09,93.33,89.21,89.92,91.75,86.14,85.35,91.49,92.12,...,92.57,93.22,93.75,92.43,93.54,94.50,90.48,95.34,96.05,97.54
frame_political,1.20,6.09,10.00,9.35,7.75,8.25,14.85,19.11,14.36,14.78,...,12.07,11.80,5.97,3.95,8.31,8.83,12.08,5.39,3.67,4.91


Reuters,2015Q1,2015Q2,2015Q3,2015Q4,2016Q1,2016Q2,2016Q3,2016Q4,2017Q1,2017Q2,...,2019Q3,2019Q4,2020Q1,2020Q2,2020Q3,2020Q4,2021Q1,2021Q2,2021Q3,2021Q4
frame_cap&res,37.50,34.48,36.59,29.21,32.04,19.48,34.64,33.94,20.63,33.18,...,34.88,35.05,36.52,33.22,28.76,27.26,28.24,40.37,53.25,54.40
frame_crime,16.67,3.45,19.51,21.35,25.24,25.32,20.26,19.39,24.87,19.43,...,14.23,15.71,10.29,7.25,15.06,14.78,15.69,12.84,3.90,7.69
frame_culture,4.17,6.90,7.32,6.74,4.37,3.25,5.23,4.24,5.29,2.84,...,4.98,6.04,4.90,4.05,3.28,3.45,4.71,2.75,0.00,3.30
frame_economic,54.17,58.62,48.78,53.93,50.97,42.21,56.86,50.30,41.27,48.82,...,52.31,55.59,50.98,49.07,46.14,44.01,46.86,58.72,76.62,76.37
frame_fairness,4.17,13.79,29.27,22.47,20.39,21.43,18.95,21.82,22.22,21.80,...,19.93,18.73,10.54,16.19,19.31,16.91,18.82,11.01,6.49,5.49
frame_health,4.17,27.59,12.20,11.24,16.02,16.88,9.15,8.48,15.34,7.58,...,17.79,15.11,42.65,45.36,27.41,31.20,31.18,22.94,16.88,11.54
frame_legality,70.83,55.17,68.29,70.79,70.39,68.83,73.20,75.76,74.07,66.35,...,65.84,64.95,54.90,57.50,65.06,66.01,60.98,56.88,45.45,56.04
frame_morality,0.00,0.00,0.00,2.25,0.00,0.65,1.31,1.21,0.00,0.00,...,0.00,0.60,0.25,0.17,0.00,0.99,0.20,0.00,0.00,0.55
frame_policy,95.83,96.55,82.93,84.27,85.92,82.47,83.66,82.42,79.89,85.78,...,87.54,88.22,84.80,84.99,85.14,85.71,85.10,90.83,93.51,96.70
frame_political,12.50,0.00,17.07,10.11,10.68,14.29,6.54,18.79,16.93,17.06,...,11.74,11.48,4.41,5.56,9.07,7.55,6.67,3.67,3.90,1.65


New York Post,2015Q1,2015Q2,2015Q3,2015Q4,2016Q1,2016Q2,2016Q3,2016Q4,2017Q1,2017Q2,...,2019Q3,2019Q4,2020Q1,2020Q2,2020Q3,2020Q4,2021Q1,2021Q2,2021Q3,2021Q4
frame_cap&res,3.53,2.80,6.67,5.22,7.02,2.20,2.84,3.92,6.25,9.16,...,4.42,3.29,3.46,5.81,4.23,5.22,4.79,3.57,6.67,5.07
frame_crime,42.35,33.64,42.67,44.78,36.84,42.86,30.50,41.18,43.75,38.17,...,43.54,46.59,35.77,31.94,32.80,32.94,43.66,35.71,34.29,37.31
frame_culture,32.94,30.84,36.00,20.90,28.07,29.67,26.24,22.55,18.75,29.77,...,22.11,22.35,21.54,15.48,20.11,18.30,18.84,27.38,22.86,20.72
frame_economic,21.18,22.43,18.67,19.40,18.42,26.37,21.99,19.61,26.56,23.66,...,15.65,14.35,14.23,13.55,12.96,16.83,15.92,19.05,16.19,16.24
frame_fairness,38.82,32.71,33.33,29.10,36.84,34.07,28.37,41.18,30.47,39.69,...,27.89,33.41,32.31,33.55,32.54,29.83,33.56,22.62,35.24,27.76
frame_health,14.12,10.28,4.00,16.42,12.28,13.19,17.73,11.76,16.41,15.27,...,22.11,20.00,26.92,45.48,30.95,29.09,24.83,25.00,20.00,25.79
frame_legality,52.94,51.40,54.67,56.72,52.63,58.24,54.61,64.71,62.50,60.31,...,61.22,58.35,60.00,55.16,58.99,59.47,68.84,47.62,52.38,58.33
frame_morality,3.53,4.67,1.33,0.75,0.00,0.00,2.84,3.92,0.78,1.53,...,2.04,2.59,1.15,4.52,4.23,1.28,2.40,0.00,0.95,1.79
frame_policy,38.82,43.93,44.00,41.04,45.61,38.46,41.84,44.12,53.12,51.91,...,42.18,44.24,51.92,56.45,51.06,52.06,48.97,45.24,58.10,51.04
frame_political,15.29,13.08,10.67,12.69,17.54,15.38,17.73,25.49,11.72,15.27,...,10.88,15.53,18.08,10.97,17.20,18.02,22.43,11.90,18.10,13.85


Breitbart News,2015Q1,2015Q2,2015Q3,2015Q4,2016Q1,2016Q2,2016Q3,2016Q4,2017Q1,2017Q2,...,2019Q3,2019Q4,2020Q1,2020Q2,2020Q3,2020Q4,2021Q1,2021Q2,2021Q3,2021Q4
frame_cap&res,3.35,4.13,3.31,3.24,1.94,2.93,2.08,1.11,1.76,3.47,...,2.83,3.77,3.40,5.25,2.69,2.10,3.35,3.30,3.20,5.03
frame_crime,43.06,38.99,51.84,52.70,44.73,43.92,50.15,42.66,42.06,44.79,...,48.18,42.06,35.50,38.98,47.83,43.46,46.78,48.52,49.28,43.67
frame_culture,19.14,19.72,19.12,14.05,16.13,19.37,14.84,16.34,12.35,14.58,...,13.77,13.89,9.62,12.37,12.41,10.61,8.58,11.83,9.60,8.89
frame_economic,11.00,12.39,11.40,8.65,6.02,9.23,5.93,9.14,10.59,9.72,...,9.51,12.90,11.54,12.20,6.13,8.05,9.65,8.35,8.32,11.09
frame_fairness,50.24,50.46,47.79,47.30,59.14,55.18,47.18,47.09,49.71,48.26,...,54.25,52.38,48.37,50.68,51.72,50.87,52.95,53.74,42.24,48.90
frame_health,7.18,11.47,5.88,5.68,5.59,4.73,8.61,5.54,6.18,6.94,...,6.68,7.14,18.05,32.03,13.15,14.91,13.14,11.65,17.44,17.59
frame_legality,72.25,71.56,73.90,72.16,71.61,73.87,76.26,70.91,79.41,80.90,...,81.58,81.94,75.44,75.42,80.27,81.06,83.51,80.52,80.32,78.94
frame_morality,2.87,8.72,6.25,6.22,5.59,7.21,9.50,7.76,5.00,3.82,...,8.91,6.15,3.55,5.25,8.22,4.39,4.69,7.13,4.00,4.89
frame_policy,49.76,52.75,50.74,52.70,48.39,53.60,46.29,49.58,54.71,57.64,...,52.63,55.16,59.32,53.73,51.12,51.88,51.21,54.43,53.44,58.52
frame_political,37.32,31.19,33.46,40.81,46.67,43.24,40.65,46.26,37.35,35.07,...,45.14,43.85,39.79,26.78,36.92,39.34,42.49,32.35,24.96,32.29


Washington Times,2015Q1,2015Q2,2015Q3,2015Q4,2016Q1,2016Q2,2016Q3,2016Q4,2017Q1,2017Q2,...,2019Q3,2019Q4,2020Q1,2020Q2,2020Q3,2020Q4,2021Q1,2021Q2,2021Q3,2021Q4
frame_cap&res,4.87,1.11,3.85,6.4,2.97,1.96,1.01,2.75,1.48,2.79,...,5.05,3.75,6.76,4.17,7.25,5.68,3.87,4.74,1.15,4.97
frame_crime,38.94,48.89,40.38,44.0,40.59,42.16,39.39,37.91,31.85,47.39,...,44.44,48.31,42.34,30.00,32.06,32.66,38.10,30.00,29.89,32.80
frame_culture,9.73,15.00,16.03,16.0,16.83,14.71,20.20,12.64,9.63,9.41,...,14.65,8.24,9.01,12.92,14.89,9.13,10.06,13.16,10.34,7.80
frame_economic,14.60,7.78,15.38,13.6,6.93,15.69,11.11,10.99,11.11,10.80,...,17.17,9.74,11.71,10.83,14.12,12.78,10.83,15.79,11.49,14.11
frame_fairness,38.05,32.78,39.74,36.0,44.55,52.94,46.46,42.31,40.74,33.10,...,47.98,50.94,38.74,38.75,45.42,43.81,40.04,44.21,51.72,53.76
frame_health,16.81,20.00,21.15,14.4,14.85,12.75,11.11,10.99,13.33,17.77,...,13.64,11.99,23.42,38.33,22.52,21.30,23.79,20.00,21.84,20.03
frame_legality,78.32,80.56,82.05,84.0,86.14,84.31,74.75,78.02,82.96,77.00,...,80.81,88.01,79.73,74.58,79.39,79.51,79.50,77.89,86.21,82.12
frame_morality,0.88,1.67,0.00,2.4,3.96,2.94,8.08,2.75,1.48,2.09,...,4.55,1.12,1.35,2.50,1.53,0.81,1.93,2.11,1.15,2.69
frame_policy,61.50,62.22,67.95,61.6,69.31,69.61,57.58,56.04,65.93,54.01,...,62.63,56.18,63.96,65.42,60.69,62.47,63.64,70.53,71.26,67.07
frame_political,18.58,17.78,22.44,21.6,38.61,38.24,39.39,34.07,32.59,16.38,...,34.85,44.94,21.17,17.50,35.88,36.31,26.69,23.16,31.03,36.29


In [10]:
# Monthly frame distribution
for outlet in outlets:
    temp = df[df["outlet_name"] == outlet].copy()
    monthly = temp.groupby("year_month")[frame_cols].mean() * 100 
    monthly.index.name = f"{outlet}"
    monthly_dict[outlet] = monthly
    display(monthly.round(2))

,frame_cap&res,frame_crime,frame_culture,frame_economic,frame_fairness,frame_health,frame_legality,frame_morality,frame_policy,frame_political,frame_public_op,frame_quality_life,frame_regulation,frame_security
HuffPost,,,,,,,,,,,,,,
2015-01,6.48,13.89,34.26,17.59,39.81,28.70,41.67,4.63,54.63,7.41,27.78,66.67,8.33,11.11
2015-02,7.41,17.59,46.30,16.67,38.89,25.00,37.04,6.48,58.33,7.41,25.00,65.74,11.11,8.33
2015-03,7.75,14.73,35.66,19.38,34.88,27.13,32.56,3.88,51.94,8.53,26.36,68.22,6.20,12.40
2015-04,12.41,16.06,35.77,26.28,40.15,20.44,34.31,3.65,62.77,15.33,32.85,67.15,13.14,12.41
2015-05,10.34,9.48,31.90,25.00,42.24,19.83,37.07,1.72,62.07,13.79,27.59,71.55,9.48,9.48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08,0.00,31.82,31.82,4.55,31.82,27.27,63.64,4.55,54.55,9.09,27.27,40.91,13.64,22.73
2021-09,5.56,33.33,5.56,16.67,55.56,22.22,77.78,0.00,83.33,27.78,33.33,27.78,16.67,33.33
2021-10,5.88,45.59,19.12,8.82,39.71,22.06,61.76,5.88,47.06,20.59,16.18,44.12,5.88,19.12


,frame_cap&res,frame_crime,frame_culture,frame_economic,frame_fairness,frame_health,frame_legality,frame_morality,frame_policy,frame_political,frame_public_op,frame_quality_life,frame_regulation,frame_security
Newsweek,,,,,,,,,,,,,,
2015-01,7.14,42.86,24.49,18.37,40.82,10.20,74.49,5.10,58.16,27.55,29.59,36.73,32.65,50.00
2015-02,5.45,36.36,32.73,16.36,38.18,12.73,68.18,4.55,51.82,20.91,28.18,40.91,34.55,49.09
2015-03,4.58,41.98,19.85,13.74,41.98,18.32,80.15,4.58,56.49,24.43,24.43,23.66,31.30,48.09
2015-04,8.46,38.46,40.00,23.85,28.46,12.31,63.08,0.77,53.08,15.38,33.85,50.77,24.62,36.15
2015-05,5.49,37.36,21.98,19.78,27.47,24.18,73.63,1.10,67.03,15.38,25.27,37.36,29.67,39.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08,3.65,32.40,28.97,6.65,29.18,28.76,55.79,1.72,62.45,10.09,30.90,56.65,16.31,34.33
2021-09,7.06,36.76,20.59,11.18,32.35,29.41,61.47,0.59,59.71,11.18,32.35,50.59,13.82,32.94
2021-10,2.81,39.12,26.89,10.27,38.75,24.21,64.30,3.30,59.66,17.36,32.03,50.24,14.55,26.89


,frame_cap&res,frame_crime,frame_culture,frame_economic,frame_fairness,frame_health,frame_legality,frame_morality,frame_policy,frame_political,frame_public_op,frame_quality_life,frame_regulation,frame_security
The Washington Post,,,,,,,,,,,,,,
2015-01,9.52,21.43,19.05,28.57,45.24,9.52,59.52,2.38,59.52,26.19,26.19,33.33,23.81,16.67
2015-02,5.41,29.73,16.22,10.81,35.14,8.11,59.46,2.70,64.86,24.32,16.22,24.32,21.62,32.43
2015-03,4.76,16.67,16.67,19.05,52.38,14.29,57.14,4.76,57.14,35.71,33.33,40.48,16.67,21.43
2015-04,6.67,26.67,11.11,20.00,51.11,13.33,51.11,2.22,53.33,31.11,31.11,31.11,15.56,22.22
2015-05,7.14,30.36,14.29,19.64,41.07,19.64,66.07,1.79,71.43,37.50,28.57,35.71,17.86,21.43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08,22.22,11.11,27.78,27.78,44.44,27.78,38.89,0.00,72.22,22.22,5.56,61.11,16.67,22.22
2021-09,29.41,14.71,8.82,44.12,29.41,23.53,32.35,0.00,50.00,26.47,17.65,52.94,8.82,17.65
2021-10,44.44,6.06,20.20,58.59,22.22,12.12,40.40,1.01,63.64,13.13,12.12,50.51,9.09,15.15


,frame_cap&res,frame_crime,frame_culture,frame_economic,frame_fairness,frame_health,frame_legality,frame_morality,frame_policy,frame_political,frame_public_op,frame_quality_life,frame_regulation,frame_security
BBC News,,,,,,,,,,,,,,
2015-01,3.23,16.13,3.23,12.90,12.90,35.48,51.61,0.00,77.42,0.00,19.35,64.52,25.81,9.68
2015-02,2.78,5.56,11.11,13.89,5.56,8.33,33.33,0.00,80.56,0.00,13.89,72.22,22.22,5.56
2015-03,0.00,0.00,31.58,21.05,5.26,10.53,36.84,0.00,68.42,10.53,31.58,84.21,21.05,15.79
2015-04,0.00,8.00,32.00,4.00,12.00,24.00,32.00,0.00,60.00,4.00,16.00,88.00,12.00,4.00
2015-05,10.00,20.00,10.00,10.00,20.00,23.33,30.00,0.00,53.33,0.00,3.33,66.67,10.00,10.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08,0.00,33.33,16.67,44.44,5.56,16.67,5.56,0.00,33.33,0.00,0.00,83.33,16.67,27.78
2021-09,0.00,33.33,66.67,66.67,0.00,0.00,33.33,33.33,33.33,0.00,33.33,66.67,0.00,33.33
2021-10,0.00,0.00,25.00,0.00,0.00,50.00,50.00,0.00,25.00,0.00,25.00,100.00,0.00,0.00


,frame_cap&res,frame_crime,frame_culture,frame_economic,frame_fairness,frame_health,frame_legality,frame_morality,frame_policy,frame_political,frame_public_op,frame_quality_life,frame_regulation,frame_security
CNBC,,,,,,,,,,,,,,
2015-01,45.00,5.00,10.00,80.00,20.00,15.00,35.00,0.0,95.00,0.00,10.00,65.00,30.00,15.00
2015-02,29.03,6.45,12.90,80.65,16.13,3.23,41.94,0.0,93.55,3.23,12.90,51.61,19.35,12.90
2015-03,46.88,6.25,15.62,96.88,9.38,12.50,37.50,0.0,96.88,0.00,3.12,56.25,18.75,15.62
2015-04,40.48,10.71,0.00,83.33,11.90,10.71,44.05,0.0,95.24,4.76,11.90,50.00,23.81,33.33
2015-05,46.25,1.25,15.00,87.50,8.75,5.00,47.50,0.0,95.00,6.25,8.75,43.75,16.25,23.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08,41.67,11.11,3.70,70.37,20.37,21.30,45.37,0.0,96.30,2.78,8.33,50.93,24.07,25.00
2021-09,33.10,8.45,5.63,66.20,19.72,21.83,53.52,0.0,95.77,4.23,9.15,56.34,18.31,20.42
2021-10,45.19,5.19,6.67,74.81,17.41,13.70,47.04,0.0,98.15,5.19,10.00,61.48,20.00,17.78


,frame_cap&res,frame_crime,frame_culture,frame_economic,frame_fairness,frame_health,frame_legality,frame_morality,frame_policy,frame_political,frame_public_op,frame_quality_life,frame_regulation,frame_security
Reuters,,,,,,,,,,,,,,
2015-01,37.50,12.50,0.00,62.50,0.00,0.00,62.50,0.00,100.00,12.50,0.00,37.50,50.00,37.50
2015-02,16.67,33.33,0.00,50.00,0.00,0.00,100.00,0.00,100.00,0.00,0.00,0.00,50.00,50.00
2015-03,50.00,10.00,10.00,50.00,10.00,10.00,60.00,0.00,90.00,20.00,0.00,20.00,70.00,40.00
2015-04,37.50,0.00,0.00,50.00,0.00,50.00,75.00,0.00,100.00,0.00,12.50,37.50,37.50,12.50
2015-05,25.00,8.33,8.33,50.00,33.33,8.33,66.67,0.00,91.67,0.00,8.33,33.33,33.33,16.67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08,62.50,6.25,0.00,75.00,12.50,18.75,31.25,0.00,87.50,0.00,0.00,56.25,31.25,18.75
2021-09,58.33,0.00,0.00,91.67,5.56,5.56,47.22,0.00,100.00,5.56,0.00,22.22,27.78,41.67
2021-10,60.42,6.25,1.04,81.25,2.08,8.33,53.12,0.00,98.96,1.04,5.21,38.54,27.08,37.50


,frame_cap&res,frame_crime,frame_culture,frame_economic,frame_fairness,frame_health,frame_legality,frame_morality,frame_policy,frame_political,frame_public_op,frame_quality_life,frame_regulation,frame_security
New York Post,,,,,,,,,,,,,,
2015-01,3.03,39.39,30.30,15.15,36.36,18.18,57.58,0.00,33.33,21.21,39.39,60.61,9.09,27.27
2015-02,9.52,38.10,47.62,28.57,57.14,9.52,38.10,9.52,47.62,4.76,14.29,57.14,0.00,19.05
2015-03,0.00,48.39,25.81,22.58,29.03,12.90,58.06,3.23,38.71,16.13,29.03,51.61,12.90,12.90
2015-04,0.00,36.36,31.82,13.64,31.82,0.00,50.00,9.09,40.91,9.09,45.45,63.64,9.09,22.73
2015-05,5.56,33.33,38.89,33.33,25.00,11.11,50.00,2.78,41.67,5.56,22.22,72.22,2.78,19.44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08,7.02,31.58,28.07,15.79,31.58,17.54,45.61,1.75,49.12,17.54,29.82,59.65,5.26,22.81
2021-09,3.33,43.33,20.00,13.33,33.33,16.67,63.33,0.00,73.33,20.00,10.00,43.33,13.33,23.33
2021-10,4.91,41.51,19.62,16.23,32.08,23.40,60.38,1.89,47.55,20.75,25.66,53.21,9.06,29.81


,frame_cap&res,frame_crime,frame_culture,frame_economic,frame_fairness,frame_health,frame_legality,frame_morality,frame_policy,frame_political,frame_public_op,frame_quality_life,frame_regulation,frame_security
Breitbart News,,,,,,,,,,,,,,
2015-01,1.96,43.14,19.61,15.69,58.82,1.96,60.78,0.00,45.10,45.10,39.22,35.29,13.73,25.49
2015-02,3.85,35.90,16.67,11.54,44.87,10.26,75.64,2.56,50.00,34.62,29.49,16.67,19.23,26.92
2015-03,3.75,50.00,21.25,7.50,50.00,7.50,76.25,5.00,52.50,35.00,33.75,16.25,32.50,46.25
2015-04,10.45,44.78,22.39,17.91,53.73,10.45,70.15,8.96,59.70,20.90,23.88,26.87,25.37,34.33
2015-05,1.85,35.19,14.81,9.26,46.30,14.81,79.63,7.41,46.30,37.04,29.63,22.22,20.37,33.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08,3.32,51.04,6.64,8.71,38.17,14.52,80.91,1.66,51.87,21.99,17.43,18.26,36.51,58.92
2021-09,2.49,45.27,7.96,6.47,44.28,18.91,81.59,5.97,54.23,27.86,20.40,14.43,31.84,51.74
2021-10,5.61,42.60,8.97,12.33,47.09,16.59,77.80,4.04,58.74,31.17,18.83,22.42,22.87,38.12


,frame_cap&res,frame_crime,frame_culture,frame_economic,frame_fairness,frame_health,frame_legality,frame_morality,frame_policy,frame_political,frame_public_op,frame_quality_life,frame_regulation,frame_security
Washington Times,,,,,,,,,,,,,,
2015-01,4.88,48.78,14.63,9.76,43.90,7.32,87.80,2.44,58.54,31.71,29.27,21.95,21.95,51.22
2015-02,1.96,33.33,13.73,11.76,37.25,21.57,72.55,0.00,60.78,11.76,23.53,35.29,19.61,27.45
2015-03,5.97,38.06,6.72,17.16,36.57,17.91,77.61,0.75,62.69,17.16,17.16,27.61,13.43,28.36
2015-04,2.44,48.78,14.63,9.76,30.49,24.39,78.05,1.22,57.32,9.76,13.41,30.49,9.76,36.59
2015-05,0.00,40.00,25.00,5.00,40.00,17.50,85.00,2.50,72.50,30.00,27.50,32.50,32.50,42.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08,0.00,17.95,10.26,5.13,53.85,23.08,79.49,0.00,76.92,33.33,20.51,15.38,20.51,46.15
2021-09,0.00,38.89,11.11,22.22,33.33,22.22,100.00,0.00,77.78,27.78,0.00,5.56,27.78,61.11
2021-10,3.48,38.26,4.35,12.17,65.22,12.17,86.09,2.61,62.61,40.00,22.61,15.65,30.43,31.30


In [11]:
# Save all to drive
annual_all = pd.concat(annual_dict, names=["outlet"])
annual_all.index = annual_all.index.set_names(["outlet", "year"])
annual_all.to_parquet("data/annual_distribution_mickey.parquet")

quarterly_all = pd.concat(quarterly_dict, names=["outlet"])
quarterly_all.index = quarterly_all.index.set_names(["outlet", "quarter"])
quarterly_all.to_parquet("data/quarterly_distribution_mickey.parquet")

monthly_all = pd.concat(monthly_dict, names=["outlet"])
monthly_all.index = monthly_all.index.set_names(["outlet", "month"])
monthly_all.to_parquet("data/monthly_distribution_mickey.parquet")